In [1]:
# Check that PyTorch and torchvision are installed, and whether a GPU is available
import torch
import torchvision
print("torch:", torch.__version__)
print("torchvision:", torchvision.__version__)
print("GPU available:", torch.cuda.is_available())

torch: 2.15.0+cpu
torchvision: 0.29.2+cpu
GPU available: False


In [2]:
# Import everything we need (plain PyTorch, no d2l library)
import time
import torch
from torch import nn
from torch.utils.data import DataLoader
import torchvision
from torchvision import transforms

# Use the GPU if there is one, otherwise the CPU
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

# Fix the random seed so results are repeatable
torch.manual_seed(0)

Using device: cpu


In [3]:
# Resize each image to 224x224 and convert it to a tensor
transform = transforms.Compose([
    transforms.Resize(224),
    transforms.ToTensor(),
])

# Download Fashion-MNIST (60,000 training images, 10,000 test images)
train_data = torchvision.datasets.FashionMNIST(root="data", train=True, transform=transform, download=True)
test_data = torchvision.datasets.FashionMNIST(root="data", train=False, transform=transform, download=True)

# Batches of 128 images; the training data is shuffled each epoch
batch_size = 128
train_loader = DataLoader(train_data, batch_size=batch_size, shuffle=True, num_workers=0)
test_loader = DataLoader(test_data, batch_size=batch_size, shuffle=False, num_workers=0)

print("Training images:", len(train_data))
print("Test images:", len(test_data))

# Look at one batch to check the shapes
images, labels = next(iter(train_loader))
print("One batch of images:", images.shape)
print("One batch of labels:", labels.shape)

Training images: 60000
Test images: 10000
One batch of images: torch.Size([128, 1, 224, 224])
One batch of labels: torch.Size([128])


In [4]:
# AlexNet in plain PyTorch (architecture from D2L Section 8.1)
alexnet = nn.Sequential(
    # Block 1: big 11x11 kernel with stride 4 shrinks the image quickly
    nn.Conv2d(1, 96, kernel_size=11, stride=4, padding=1), nn.ReLU(),
    nn.MaxPool2d(kernel_size=3, stride=2),

    # Block 2: 5x5 kernel, padding 2 keeps the size, 256 channels
    nn.Conv2d(96, 256, kernel_size=5, padding=2), nn.ReLU(),
    nn.MaxPool2d(kernel_size=3, stride=2),

    # Block 3: three 3x3 conv layers in a row (384, 384, 256 channels)
    nn.Conv2d(256, 384, kernel_size=3, padding=1), nn.ReLU(),
    nn.Conv2d(384, 384, kernel_size=3, padding=1), nn.ReLU(),
    nn.Conv2d(384, 256, kernel_size=3, padding=1), nn.ReLU(),
    nn.MaxPool2d(kernel_size=3, stride=2),

    # Turn the 256 x 5 x 5 feature maps into one long vector of 6400 numbers
    nn.Flatten(),

    # Three fully connected layers; dropout randomly switches off half the
    # neurons during training so the model does not overfit
    nn.Linear(6400, 4096), nn.ReLU(), nn.Dropout(p=0.5),
    nn.Linear(4096, 4096), nn.ReLU(), nn.Dropout(p=0.5),
    nn.Linear(4096, 10),   # 10 clothing classes
)

print("AlexNet created.")

AlexNet created.


In [5]:
# Send one fake image (1 image, 1 channel, 224 x 224) through the model
# and print the output shape after every layer
X = torch.randn(1, 1, 224, 224)
for layer in alexnet:
    X = layer(X)
    print(f"{layer.__class__.__name__:<12} output shape: {tuple(X.shape)}")

Conv2d       output shape: (1, 96, 54, 54)
ReLU         output shape: (1, 96, 54, 54)
MaxPool2d    output shape: (1, 96, 26, 26)
Conv2d       output shape: (1, 256, 26, 26)
ReLU         output shape: (1, 256, 26, 26)
MaxPool2d    output shape: (1, 256, 12, 12)
Conv2d       output shape: (1, 384, 12, 12)
ReLU         output shape: (1, 384, 12, 12)
Conv2d       output shape: (1, 384, 12, 12)
ReLU         output shape: (1, 384, 12, 12)
Conv2d       output shape: (1, 256, 12, 12)
ReLU         output shape: (1, 256, 12, 12)
MaxPool2d    output shape: (1, 256, 5, 5)
Flatten      output shape: (1, 6400)
Linear       output shape: (1, 4096)
ReLU         output shape: (1, 4096)
Dropout      output shape: (1, 4096)
Linear       output shape: (1, 4096)
ReLU         output shape: (1, 4096)
Dropout      output shape: (1, 4096)
Linear       output shape: (1, 10)


In [6]:
# Start the weights with Xavier initialization (as in the book)
def init_weights(layer):
    if isinstance(layer, (nn.Linear, nn.Conv2d)):
        nn.init.xavier_uniform_(layer.weight)

alexnet.apply(init_weights)
alexnet.to(device)

# Loss function and optimizer (settings from the task: SGD, learning rate 0.01)
loss_fn = nn.CrossEntropyLoss()
optimizer = torch.optim.SGD(alexnet.parameters(), lr=0.01)


def evaluate(model, loader):
    '''Accuracy of the model on a data loader (no learning happens here).'''
    model.eval()                       # switch dropout off
    correct, total = 0, 0
    with torch.no_grad():              # we do not need gradients for testing
        for X, y in loader:
            X, y = X.to(device), y.to(device)
            correct += (model(X).argmax(dim=1) == y).sum().item()
            total += y.size(0)
    return correct / total


def train(model, epochs):
    '''Train the model; print and return the loss and test accuracy of each epoch.'''
    history = []
    for epoch in range(epochs):
        model.train()                  # switch dropout on
        total_loss, count, start = 0.0, 0, time.time()
        for X, y in train_loader:
            X, y = X.to(device), y.to(device)
            optimizer.zero_grad()      # clear old gradients
            loss = loss_fn(model(X), y)
            loss.backward()            # backpropagation
            optimizer.step()           # update the weights
            total_loss += loss.item() * y.size(0)
            count += y.size(0)
        train_loss = total_loss / count
        test_acc = evaluate(model, test_loader)
        history.append((epoch + 1, train_loss, test_acc))
        print(f"epoch {epoch + 1:2d} | train loss {train_loss:.4f} | test acc {test_acc:.4f} | {time.time() - start:.0f}s")
    return history

print("Training code is ready.")

Training code is ready.


In [7]:
# Train AlexNet for 10 epochs (settings: SGD, lr 0.01, batch size 128)
alexnet.apply(init_weights)            # start from fresh weights
history = train(alexnet, epochs=10)    # prints loss and test accuracy of each epoch

# Save the results to a text file, so we keep a copy of the numbers
with open("alexnet_results.csv", "w") as f:
    f.write("epoch,train_loss,test_accuracy\n")
    for epoch, loss, acc in history:
        f.write(f"{epoch},{loss:.4f},{acc:.4f}\n")
print("Saved alexnet_results.csv")

epoch  1 | train loss 1.4063 | test acc 0.7305 | 1464s
epoch  2 | train loss 0.6528 | test acc 0.7900 | 1510s
epoch  3 | train loss 0.5382 | test acc 0.8271 | 1504s
epoch  4 | train loss 0.4718 | test acc 0.8386 | 1535s
epoch  5 | train loss 0.4274 | test acc 0.8528 | 1709s
epoch  6 | train loss 0.3978 | test acc 0.8584 | 1678s
epoch  7 | train loss 0.3745 | test acc 0.8651 | 1344s
epoch  8 | train loss 0.3568 | test acc 0.8715 | 1362s
epoch  9 | train loss 0.3410 | test acc 0.8790 | 1368s
epoch 10 | train loss 0.3285 | test acc 0.8852 | 3209s
Saved alexnet_results.csv


## How AlexNet differs from LeNet and why those changes help

AlexNet is much deeper and bigger than LeNet: it has 5 convolutional layers and 3 fully connected layers (8 layers in total), 96 to 384 channels instead of 6 and 16, and about 47 million parameters in our version (about 60 million in the original ImageNet version), so it can learn richer features from larger images (224×224 instead of 28×28). It uses ReLU activations instead of sigmoid; ReLU has a gradient of 1 for positive inputs and does not saturate, so the gradient does not vanish and the network trains faster and more easily. It uses max pooling instead of average pooling, which keeps the strongest feature responses and is more tolerant to small shifts of the object. It adds dropout (p = 0.5) in the two large fully connected layers, which randomly switches off neurons during training and reduces overfitting, and it starts with a large 11×11 kernel with stride 4 to quickly shrink big images. These changes, together with training on GPUs and large amounts of data, are what made AlexNet much more accurate than LeNet on real images.

In [8]:
# Save the trained weights (the learned numbers of every layer) to a file
torch.save(alexnet.state_dict(), "alexnet_fashion_mnist.pth")
print("Saved alexnet_fashion_mnist.pth")

# To load them again later, you would build the same model and run:
# alexnet.load_state_dict(torch.load("alexnet_fashion_mnist.pth"))

Saved alexnet_fashion_mnist.pth
